# Frozen ModernBERT

**Question (Further Work item 3):** does a newer encoder give better frozen embeddings than RoBERTa? ModernBERT-base (Dec 2024) is a BERT-style encoder like RoBERTa: bidirectional, trained with masked language modeling, with a 768-d output. It is newer in architecture and data: 22 layers instead of 12 (149M parameters vs. 125M), 2 trillion training tokens of web text, code and papers, and an 8,192-token context.

Same data and code as `Frozen Roberta MLP.ipynb` (full 20 Newsgroups, same 90/10 split), with two changes:
1. **The encoder:** `answerdotai/ModernBERT-base` instead of `roberta-base`.
2. **The classifier settings:** the best RoBERTa settings from the hyperparameter search (`hparam_search/MLPHyperparameterSearch.md`) instead of the first notebook's guesses. They weren't tuned for ModernBERT, so they're an educated guess here.

**Predict before running:** with these settings, RoBERTa got 66.77 (LR) and 68.33 (MLP) on test. Will ModernBERT beat it?

## Load dataset (copied from the PLAYGROUND)

In [1]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_20newsgroups

newsgroups_train = fetch_20newsgroups(
    subset='train', remove=('headers', 'footers', 'quotes'))

newsgroups_test = fetch_20newsgroups(
    subset='test', remove=('headers', 'footers', 'quotes'))

# Remove empty documents
for s, name in zip([newsgroups_train, newsgroups_test], ["train", "test"]):
  empty_indices = {i for i, doc in enumerate(s.data) if len(doc) == 0}
  orig_len = len(s.data)
  for k in ['data', 'filenames', 'target']:
    s[k] = [s[k][i] for i in range(orig_len) if i not in empty_indices]
  print(f"Removed {len(empty_indices)} empty documents from the {name} set. Before: {orig_len}. After: {len(s.data)}.")

y_train = np.array(newsgroups_train.target)
y_test = np.array(newsgroups_test.target)
n_classes = len(newsgroups_train.target_names)

Removed 218 empty documents from the train set. Before: 11314. After: 11096.
Removed 162 empty documents from the test set. Before: 7532. After: 7370.


## ModernBERT `[CLS]` embeddings

The same code as the RoBERTa notebook, with `answerdotai/ModernBERT-base` in place of `roberta-base`. ModernBERT uses BERT's special tokens, so position 0 is `[CLS]` instead of `<s>`, in the same role. Two choices keep the encoder as the only change:
- **Position 0, not an average over all tokens.** ModernBERT's own fine-tuning class (`ModernBertForSequenceClassification`) averages instead, because its config sets `classifier_pooling: "mean"`. Mean pooling may suit it better; that's a separate experiment.
- **The first 512 tokens**, as for RoBERTa, though ModernBERT can read 8,192 (longer inputs are Further Work item 4).

The result is saved to `modernbert_cls_20ng.npz`, so later runs skip this cell's slow part (about 3 minutes on the GPU).

In [2]:
import os
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
CACHE = "modernbert_cls_20ng.npz"

## X is a list of strings (messages). Returns a np 2D array of shape (len(X), 768):
## the [CLS] embedding of each message, like RobertaVectorizer.transform in A2
def modernbert_embed(X, tokenizer, modernbert, batch_size=32):
    vecs = []
    for i in tqdm(range(0, len(X), batch_size)):
        batch = tokenizer(X[i:i + batch_size], padding=True, truncation=True, max_length=512,
                          return_tensors="pt").to(device)
        with torch.no_grad():  # frozen: no gradients needed
            out = modernbert(**batch)
        # [:, 0] = token 0 ([CLS]) of every message in the batch, like RoBERTa's <s>
        vecs.append(out.last_hidden_state[:, 0].cpu().numpy())
    return np.concatenate(vecs)

if os.path.exists(CACHE):
    saved = np.load(CACHE)
    X_train, X_test = saved["X_train"], saved["X_test"]
else:
    tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    # No add_pooling_layer=False this time: ModernBERT has no pooler
    modernbert = AutoModel.from_pretrained("answerdotai/ModernBERT-base").to(device).eval()
    X_train = modernbert_embed(newsgroups_train.data, tokenizer, modernbert)
    X_test = modernbert_embed(newsgroups_test.data, tokenizer, modernbert)
    np.savez(CACHE, X_train=X_train, X_test=X_test)

print(X_train.shape, X_test.shape)

/home/tonyxdsu/venvs/cs423/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(11096, 768) (7370, 768)


## Baseline: logistic regression

First, hold out 10% of train as a dev set, the same split as the RoBERTa notebooks. The MLP needs it for early stopping; the test set is only used at the very end. LR trains on the same 90%, so the comparison stays fair.

LR uses the RoBERTa search's best settings:
- **Standardized inputs** (mean 0 and std 1 in every dimension, from the 90%'s statistics only). ModernBERT's `[CLS]` values spread about 15× wider than RoBERTa's `<s>` values (median per-dimension std 0.38 vs. 0.025). On raw inputs, the same `C` would mean a different amount of regularization. Standardized, both encoders are on the same scale, so RoBERTa's best `C` is a fair guess.
- **`C = 0.003`**, the best for standardized RoBERTa features (70.78 CV, 66.77 test).
- **1 BLAS thread** (`threadpool_limits(1)`), which is about 60× faster on this machine.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from threadpoolctl import threadpool_limits

X_tr, X_dev, y_tr, y_dev = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=423)

scaler = StandardScaler().fit(X_tr)  # mean and std of the 90% only, so dev and test stay unseen
X_tr, X_dev = scaler.transform(X_tr), scaler.transform(X_dev)

with threadpool_limits(1):
    logreg = LogisticRegression(C=0.003, max_iter=1000).fit(X_tr, y_tr)
lr_acc = metrics.accuracy_score(y_test, logreg.predict(scaler.transform(X_test)))
print(f"LR test accuracy: {lr_acc*100:.2f}")

LR test accuracy: 55.51


## MLP

768 → 2048 → 64 → 20, the best RoBERTa config from the search (the first notebook used 1024 → 256). The search found that these mattered:
- **A small learning rate:** AdamW at lr 1e-4. The first notebook's 1e-3 is too big a step once inputs are standardized. Weight decay stays at AdamW's default, 0.01.
- **Heavy dropout in every place:** 0.2 on the input, 0.7 after hidden layer 1, 0.5 after hidden layer 2. Without dropout, the MLP was only level with LR.
- **Label smoothing 0.1:** the training target puts 90% of the probability on the right class and spreads the other 10% evenly over all classes, so the model is never pushed toward 100% confidence.

GELU and batch size 32 also come from the search, but barely mattered there.

In [4]:
torch.manual_seed(423)

mlp = nn.Sequential(
    nn.Dropout(0.2),                                   # dropout on the input embedding
    nn.Linear(768, 2048), nn.GELU(), nn.Dropout(0.7),  # hidden layer 1: wide, heavy dropout
    nn.Linear(2048, 64), nn.GELU(), nn.Dropout(0.5),   # hidden layer 2: narrow
    nn.Linear(64, n_classes),                          # one logit per class
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-4)  # weight decay: AdamW's default, 0.01

Training loop with early stopping, as in the A1 search: stop once dev loss hasn't improved for `PATIENCE` epochs, then go back to the best epoch's weights. Two changes from the first notebook, both from the search:
- **Patience 10 and up to 300 epochs** (was 5 and 100). A 10× smaller learning rate needs more epochs: at lr 1e-4, RoBERTa's best epoch was around 50–90.
- **Dev loss is plain cross-entropy, without label smoothing.** The train loss includes smoothing, so it can't go below about 0.6 and isn't directly comparable to the dev loss.

In [5]:
import copy
import time
from torch.utils.data import TensorDataset, DataLoader

loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=32, shuffle=True)
X_dev_t, y_dev_t = torch.tensor(X_dev).to(device), torch.tensor(y_dev).to(device)

PATIENCE = 10
best = {"dev_loss": float("inf"), "epoch": 0, "state": None}
start = time.perf_counter()

for epoch in range(1, 301):
    mlp.train()  # dropout on
    train_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)

    mlp.eval()  # dropout off
    with torch.no_grad():
        dev_logits = mlp(X_dev_t)
        dev_loss = nn.functional.cross_entropy(dev_logits, y_dev_t).item()  # no label smoothing, as in the search
        dev_acc = (dev_logits.argmax(dim=1) == y_dev_t).float().mean().item()
    print(f"epoch {epoch:3d}  train loss {train_loss / len(X_tr):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}")

    if dev_loss < best["dev_loss"]:
        best = {"dev_loss": dev_loss, "epoch": epoch, "state": copy.deepcopy(mlp.state_dict())}
    elif epoch - best["epoch"] >= PATIENCE:
        break

mlp.load_state_dict(best["state"])
print(f"Training took {time.perf_counter() - start:.1f} s. Using epoch {best['epoch']} (lowest dev loss)")

epoch   1  train loss 2.815  dev loss 2.488  dev acc 29.37


epoch   2  train loss 2.509  dev loss 2.166  dev acc 37.48


epoch   3  train loss 2.338  dev loss 1.986  dev acc 43.15


epoch   4  train loss 2.231  dev loss 1.866  dev acc 45.95


epoch   5  train loss 2.146  dev loss 1.781  dev acc 48.47


epoch   6  train loss 2.110  dev loss 1.726  dev acc 50.18


... (86 outputs left out)

epoch  93  train loss 1.548  dev loss 1.320  dev acc 60.09


epoch  94  train loss 1.546  dev loss 1.319  dev acc 59.01


epoch  95  train loss 1.541  dev loss 1.311  dev acc 60.00


epoch  96  train loss 1.544  dev loss 1.310  dev acc 58.92
Training took 222.0 s. Using epoch 86 (lowest dev loss)


## LR vs. MLP on the test set

In [6]:
mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(torch.tensor(scaler.transform(X_test)).to(device)).argmax(dim=1).cpu().numpy()
mlp_acc = metrics.accuracy_score(y_test, mlp_pred)

print(f"LR  test accuracy: {lr_acc*100:.2f}")
print(f"MLP test accuracy: {mlp_acc*100:.2f}")
print(f"Difference: {(mlp_acc - lr_acc)*100:+.2f} points (1 point = {len(y_test) / 100:.0f} test messages)")

LR  test accuracy: 55.51
MLP test accuracy: 57.75
Difference: +2.24 points (1 point = 74 test messages)


For comparison, RoBERTa with the same settings (`hparam_search/final_test.json`): **LR 66.77, MLP 68.33 ± 0.17** over 10 seeds.

This is one MLP with one seed, and the settings were tuned for RoBERTa, not ModernBERT. A different seed shifts the result a little, so read a gap under about a point as "about the same".

# Mean pooling + tuned classifiers

Added after the hyperparameter search on mean-pooled features (`hparam_search_mean/`). Two things change from the sections above:
1. **Mean pooling:** each message's vector is the average of all its token vectors, instead of only `[CLS]`. ModernBERT's own classification head pools this way, and a quick check showed it helps ModernBERT a lot.
2. **Tuned settings**, chosen by 5-fold cross-validation on train. Nothing was chosen by looking at the test set.

Same data and 90/10 split as above, so the numbers are comparable.

In [7]:
CACHE_MEAN = "modernbert_mean_20ng.npz"

## Same as modernbert_embed above, except that it averages the vectors of all of a message's tokens (mean pooling)
## instead of keeping only [CLS]. Padding is left out of the average.
def modernbert_embed_mean(X, tokenizer, modernbert, batch_size=32):
    vecs = []
    for i in tqdm(range(0, len(X), batch_size)):
        batch = tokenizer(X[i:i + batch_size], padding=True, truncation=True, max_length=512,
                          return_tensors="pt").to(device)
        with torch.no_grad():  # frozen: no gradients needed
            out = modernbert(**batch)
        mask = batch["attention_mask"].unsqueeze(-1)  # (batch, tokens, 1): 1 = real token, 0 = padding
        vecs.append(((out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)).cpu().numpy())
    return np.concatenate(vecs)

if os.path.exists(CACHE_MEAN):
    saved = np.load(CACHE_MEAN)
    X_train_mean, X_test_mean = saved["X_train"], saved["X_test"]
else:
    tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    modernbert = AutoModel.from_pretrained("answerdotai/ModernBERT-base").to(device).eval()
    X_train_mean = modernbert_embed_mean(newsgroups_train.data, tokenizer, modernbert)
    X_test_mean = modernbert_embed_mean(newsgroups_test.data, tokenizer, modernbert)
    np.savez(CACHE_MEAN, X_train=X_train_mean, X_test=X_test_mean)

print(X_train_mean.shape, X_test_mean.shape)

(11096, 768) (7370, 768)


## Logistic regression, tuned

Raw inputs and `C = 0.1`: the best of a `C` grid (1e-4 to 1e3) with and without standardizing, at 68.62 CV accuracy. The `StandardScaler` is fit on the 90% only, as above; the MLP below uses the same standardized inputs.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

X_tr_mean, X_dev_mean, y_tr, y_dev = train_test_split(
    X_train_mean, y_train, test_size=0.1, stratify=y_train, random_state=423)
X_tr_raw = X_tr_mean  # LR uses the raw vectors; the MLP the standardized ones

scaler_mean = StandardScaler().fit(X_tr_mean)  # mean and std of the 90% only, so dev and test stay unseen
X_tr_mean, X_dev_mean = scaler_mean.transform(X_tr_mean), scaler_mean.transform(X_dev_mean)

with threadpool_limits(1):  # 1 BLAS thread: much faster on this machine
    logreg_mean = LogisticRegression(C=0.1, max_iter=5000).fit(X_tr_raw, y_tr)
lr_mean_acc = metrics.accuracy_score(y_test, logreg_mean.predict(X_test_mean))
print(f"LR test accuracy: {lr_mean_acc*100:.2f}")

LR test accuracy: 64.87


## MLP, tuned

768 → 2048 → 128 → 20: the config with the best 3-seed CV accuracy in the search (70.92).
- **Dropout:** 0.2 on the input, 0.7 after hidden layer 1, 0.6 after hidden layer 2.
- **AdamW** at lr 1e-4, weight decay 0; **label smoothing** 0.1.
- GELU, batch size 32, standardized inputs, early stopping on dev loss with patience 10 (up to 300 epochs), as in the search.

In [9]:
torch.manual_seed(423)

mlp_mean = nn.Sequential(
    nn.Dropout(0.2),                                   # dropout on the input embedding
    nn.Linear(768, 2048), nn.GELU(), nn.Dropout(0.7),  # hidden layer 1
    nn.Linear(2048, 128), nn.GELU(), nn.Dropout(0.6),  # hidden layer 2
    nn.Linear(128, n_classes),                         # one logit per class
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(mlp_mean.parameters(), lr=1e-4, weight_decay=0)

Same training loop as the search: early stopping on plain cross-entropy dev loss, patience 10, up to 300 epochs.

In [10]:
import copy
import time
from torch.utils.data import TensorDataset, DataLoader

loader = DataLoader(TensorDataset(torch.tensor(X_tr_mean), torch.tensor(y_tr)), batch_size=32, shuffle=True)
X_dev_t, y_dev_t = torch.tensor(X_dev_mean).to(device), torch.tensor(y_dev).to(device)

PATIENCE = 10
best = {"dev_loss": float("inf"), "epoch": 0, "state": None}
start = time.perf_counter()

for epoch in range(1, 301):
    mlp_mean.train()  # dropout on
    train_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp_mean(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)

    mlp_mean.eval()  # dropout off
    with torch.no_grad():
        dev_logits = mlp_mean(X_dev_t)
        dev_loss = nn.functional.cross_entropy(dev_logits, y_dev_t).item()  # no label smoothing, as in the search
        dev_acc = (dev_logits.argmax(dim=1) == y_dev_t).float().mean().item()
    print(f"epoch {epoch:3d}  train loss {train_loss / len(X_tr_mean):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}")

    if dev_loss < best["dev_loss"]:
        best = {"dev_loss": dev_loss, "epoch": epoch, "state": copy.deepcopy(mlp_mean.state_dict())}
    elif epoch - best["epoch"] >= PATIENCE:
        break

mlp_mean.load_state_dict(best["state"])
print(f"Training took {time.perf_counter() - start:.1f} s. Using epoch {best['epoch']} (lowest dev loss)")

epoch   1  train loss 2.552  dev loss 1.834  dev acc 53.96


epoch   2  train loss 1.970  dev loss 1.440  dev acc 61.80


epoch   3  train loss 1.812  dev loss 1.319  dev acc 65.05


epoch   4  train loss 1.720  dev loss 1.253  dev acc 65.77


epoch   5  train loss 1.660  dev loss 1.205  dev acc 66.04


epoch   6  train loss 1.619  dev loss 1.168  dev acc 66.76


... (41 outputs left out)

epoch  48  train loss 1.228  dev loss 0.997  dev acc 70.99


epoch  49  train loss 1.228  dev loss 0.994  dev acc 70.54


epoch  50  train loss 1.232  dev loss 0.998  dev acc 70.54


epoch  51  train loss 1.218  dev loss 0.995  dev acc 70.36
Training took 35.0 s. Using epoch 41 (lowest dev loss)


## `[CLS]` vs. mean pooling on the test set

In [11]:
mlp_mean.eval()
with torch.no_grad():
    mlp_mean_pred = mlp_mean(torch.tensor(scaler_mean.transform(X_test_mean)).to(device)).argmax(dim=1).cpu().numpy()
mlp_mean_acc = metrics.accuracy_score(y_test, mlp_mean_pred)

rows = [("[CLS], RoBERTa's settings (above)", lr_acc, mlp_acc),
        ("mean pooling, tuned settings", lr_mean_acc, mlp_mean_acc)]
print(f"{'test accuracy':<42}{'LR':>7}{'MLP':>8}")
for name, lr_a, mlp_a in rows:
    print(f"{name:<42}{lr_a*100:>7.2f}{mlp_a*100:>8.2f}")
print(f"(1 point = {len(y_test) / 100:.0f} test messages)")

test accuracy                                  LR     MLP
[CLS], RoBERTa's settings (above)           55.51   57.75
mean pooling, tuned settings                64.87   66.38
(1 point = 74 test messages)


The search's own final test retrained this MLP config with 10 seeds on the same split: **66.28 ± 0.18** (`hparam_search_mean/final_test.json`), so read the single seed above against that. For reference, RoBERTa `<s>` with its tuned settings got LR 66.77 and MLP 68.33 ± 0.17 (`hparam_search/final_test.json`).

Everything was tuned by CV on train; the test set was only used to score the final settings (here and in `final_test.json`).